# backtesting with geo model

In [2]:
import pandas as pd
import tensorflow as tf
import h3
import numpy as np
import h3pandas
import pydeck as pdk
from tqdm.auto import tqdm
from shapely.geometry import Polygon


2025-09-04 10:04:48.117712: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-09-04 10:04:48.117738: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-04 10:04:48.118711: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-04 10:04:48.123390: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-04 10:04:48.679870: W tensorflow/compiler/tf2

In [3]:
import sys
sys.path.append("../")
from lib.models.geo_model_net import ResLayer
from lib.models.encoders import CoordEncoder

In [4]:
class ResLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(ResLayer, self).__init__()
        self.w1 = tf.keras.layers.Dense(
            256,
            activation="relu",
        )
        self.w2 = tf.keras.layers.Dense(
            256,
            activation="relu",
        )
        self.dropout = tf.keras.layers.Dropout(rate=0.5)
        self.add = tf.keras.layers.Add()

    def call(self, inputs):
        x = self.w1(inputs)
        x = self.dropout(x)
        x = self.w2(x)
        x = self.add([x, inputs])
        return x

    def get_config(self):
        return {}

In [5]:
geomodel_2_20 = tf.keras.models.load_model(
    "saved_models/saved_model_2_20_coords_elev.h5",
    custom_objects={"ResLayer": ResLayer},
    compile=False,
)

2025-09-04 10:04:52.640423: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-09-04 10:04:52.671701: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-09-04 10:04:52.673145: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [6]:
!ls /data-ssd/alex/datasets/vision-export-20250119070012-aka-2.20/

all_taxa.csv	  spatial_data.parquet	test_data.csv
iconic_taxa.csv   taxonomy.csv		train_data.csv
spatial_data.csv  taxonomy_visual.txt	val_data.csv


In [7]:
tax_2_20 = pd.read_csv(
    "/data-ssd/alex/datasets/vision-export-20250119070012-aka-2.20/taxonomy.csv"
)

In [8]:
res0 = h3.get_res0_indexes()  # 122 res-0 cells
h3_indices = []
for h in res0:
    h3_indices.extend(h3.h3_to_children(h, 4))
len(h3_indices)

288122

In [9]:
df = pd.DataFrame({
    "h3_04": list(h3_indices)
}).set_index("h3_04")
dfh3 = df.h3.h3_to_geo()
dfh3["lng"] = dfh3.geometry.x
dfh3["lat"] = dfh3.geometry.y
_ = dfh3.pop("geometry")

In [10]:
raster = np.load("/home/alex/sinr/data/env/elev_scaled.npy")
encoder = CoordEncoder(
    encoding_strategy="sinusoidal",
    raster=raster
)

In [11]:
locs = dfh3.to_numpy()
encoded_locs = encoder.encode(locs)

In [12]:
class_of_interest = int(tax_2_20[tax_2_20["name"] == "Crotalus ruber"].iloc[0].leaf_class_id)
taxon_of_interest = int(tax_2_20[tax_2_20["name"] == "Crotalus ruber"].iloc[0].taxon_id)

class_of_interest, taxon_of_interest

(85773, 30724)

In [13]:
thresholds = pd.read_csv(
    "/home/alex/inatVisionApi/geo_model_eval_output/v2_20_sinr/thresholds.csv"
)

In [41]:
threshold = thresholds[thresholds.taxon_id==taxon_of_interest].iloc[0].thres
#threshold = 0.001
threshold

0.09890909

In [42]:
loc_emb = geomodel_2_20.layers[0](encoded_locs)

# res layers - feature extraction
x = geomodel_2_20.layers[1](loc_emb)
x = geomodel_2_20.layers[2](x)
x = geomodel_2_20.layers[3](x)
geo_model_feats = geomodel_2_20.layers[4](x)

raw_preds = tf.matmul(
    tf.expand_dims(geomodel_2_20.layers[5].weights[0][:, class_of_interest], axis=0),
    geo_model_feats,
    transpose_b=True
)
preds = tf.keras.activations.sigmoid(raw_preds).numpy()

In [43]:
dfh3["score"] = preds[0]
predicted_cells = list(dfh3[dfh3.score>threshold].index)
predicted_cells[:5]

['84480d3ffffffff',
 '84485c3ffffffff',
 '8448005ffffffff',
 '84485d5ffffffff',
 '8448555ffffffff']

# ok let's load up the 2.24 training data and look at stuff seen SINCE 2.20 and see what proportion are in and out of the expected range

In [17]:
train_data_2_24 = pd.read_parquet(
    "/data-ssd/alex/datasets/vision-export-20250803060005-aka-2.24/spatial_data.parquet"
)

In [18]:
train_data_2_24.dropna(subset="observed_on", inplace=True)

In [19]:
train_data_2_24.observed_on = pd.to_datetime(train_data_2_24.observed_on, errors="coerce")

In [20]:
train_data_2_24.dropna(subset="observed_on", inplace=True)

In [21]:
train_data_2_24_subset = train_data_2_24[
    train_data_2_24.taxon_id==taxon_of_interest
]

In [22]:
train_data_2_24_old_subset = train_data_2_24_subset[
    (train_data_2_24_subset.observed_on.dt.year<=2024)
]

In [23]:
train_data_2_24_new_subset = train_data_2_24_subset[
    (train_data_2_24_subset.observed_on.dt.year>2024)
]

In [24]:
train_data_2_24_new_subset

,observation_id,latitude,longitude,observed_on,leaf_class_id,iconic_class_id,spatial_class_id,taxon_id,obs_taxon_id,community,captive,positional_accuracy
54125052,296429759,33.9936,-116.6587,2025-07-08,28035.0,3,28035,30724,30724,1,0,4.0
54125066,285342944,32.7136,-116.9183,2025-05-25,28035.0,3,28035,30724,30724,1,0,3.0
54125071,290794355,34.0528,-116.5952,2025-05-01,28035.0,3,28035,30724,30724,1,0,2.0
54125078,284866634,33.2135,-116.4256,2025-05-25,28035.0,3,28035,30724,30724,1,0,4.0
54125086,269889579,33.0069,-116.9886,2025-04-11,28035.0,3,28035,30724,30724,1,0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...
54129864,270530690,32.7322,-117.1477,2025-04-11,28035.0,3,28035,30724,30724,0,1,121.0
54129872,291096978,43.9907,-103.2720,2025-06-13,28035.0,3,28035,30724,30724,0,1,32.0
54129876,291630176,38.1401,-122.2367,2025-06-21,28035.0,3,28035,30724,30724,0,1,36.0
54129878,301519914,52.0087,5.9003,2025-06-23,28035.0,3,28035,30724,30724,0,1,15.0


In [25]:
train_data_2_24_new_subset_h3 = train_data_2_24_new_subset.h3.geo_to_h3(
    resolution=4,
    lat_col="latitude",
    lng_col="longitude",
)
train_data_2_24_new_subset_h3.reset_index(inplace=True)
train_data_2_24_new_subset_h3.drop_duplicates(subset=["h3_04"], inplace=True)

In [26]:
train_data_2_24_old_subset_h3 = train_data_2_24_old_subset.h3.geo_to_h3(
    resolution=4,
    lat_col="latitude",
    lng_col="longitude",
)
train_data_2_24_old_subset_h3.reset_index(inplace=True)
train_data_2_24_old_subset_h3.drop_duplicates(subset=["h3_04"], inplace=True)

In [27]:
new_occurred = list(train_data_2_24_new_subset_h3.h3_04.values)
new_occurred[:5]

['8429a05ffffffff',
 '8429a4dffffffff',
 '8429a6bffffffff',
 '8429a45ffffffff',
 '8429a01ffffffff']

In [28]:
old_occurred = list(train_data_2_24_old_subset_h3.h3_04.values)
old_occurred[:5]

['8429a41ffffffff',
 '8429a63ffffffff',
 '8429a47ffffffff',
 '8429a09ffffffff',
 '844823bffffffff']

In [29]:
true_new = list(set(new_occurred) - set(old_occurred))
true_new

['8426165ffffffff', '8444de9ffffffff']

# do the loop

In [44]:
stable_negatives = 0
stable_positives = 0
discovery_misses = 0
discovery_hits = 0
pred_not_found = 0
found_not_pred = 0

for cell in tqdm(dfh3.index.values):
    in_pred = cell in predicted_cells
    in_new = cell in true_new
    in_old = cell in old_occurred

    if in_new:
        if in_pred:
            discovery_hits += 1
        else:
            discovery_misses += 1
    elif in_old:
        if in_pred:
            stable_positives += 1
        else:
            found_not_pred += 1
    else:
        if in_pred:
            pred_not_found += 1
        else:
            stable_negatives += 1
    if cell not in predicted_cells and cell not in new_occurred:
        stable_negatives += 1

print(f"{stable_negatives} stable negatives")
print(f"{stable_positives} stable positives")
print()
print(f"{pred_not_found} expected but no occurrence")
print(f"{found_not_pred} occurence but not expected")
print()
print(f"{discovery_hits} discovery hits")
print(f"{discovery_misses} discovery misses")

  0%|          | 0/288122 [00:00<?, ?it/s]

575928 stable negatives
72 stable positives

65 expected but no occurrence
31 occurence but not expected

0 discovery hits
2 discovery misses


# both the "discoveries" are captive and way out of range :/

In [45]:
train_data_2_24_subset_h3 = train_data_2_24_subset.h3.geo_to_h3(
    resolution=4,
    lat_col="latitude",
    lng_col="longitude",
)
train_data_2_24_subset_h3.reset_index(inplace=True)
train_data_2_24_subset_h3.drop_duplicates(subset=["h3_04"], inplace=True)

In [46]:
# south dakota
train_data_2_24_subset_h3[
    train_data_2_24_subset_h3.h3_04=="8426165ffffffff"
]

,h3_04,observation_id,latitude,longitude,observed_on,leaf_class_id,iconic_class_id,spatial_class_id,taxon_id,obs_taxon_id,community,captive,positional_accuracy
4801,8426165ffffffff,291096978,43.9907,-103.272,2025-06-13,28035.0,3,28035,30724,30724,0,1,32.0


In [47]:
# south carolina
train_data_2_24_subset_h3[
    train_data_2_24_subset_h3.h3_04=="8444de9ffffffff"
]

,h3_04,observation_id,latitude,longitude,observed_on,leaf_class_id,iconic_class_id,spatial_class_id,taxon_id,obs_taxon_id,community,captive,positional_accuracy
4816,8444de9ffffffff,276420773,34.0087,-81.0727,2025-04-27,28035.0,3,28035,30724,30724,0,1,30.0


# let's evaluate with only non-captive data with a CID

In [49]:
train_data_2_24_subset = train_data_2_24_subset[
    (train_data_2_24_subset.captive==0)&(train_data_2_24_subset.community==1)
]

In [51]:
train_data_2_24_old_subset = train_data_2_24_subset[
    (train_data_2_24_subset.observed_on.dt.year<=2024)
]
train_data_2_24_new_subset = train_data_2_24_subset[
    (train_data_2_24_subset.observed_on.dt.year>2024)
]

In [52]:
train_data_2_24_new_subset_h3 = train_data_2_24_new_subset.h3.geo_to_h3(
    resolution=4,
    lat_col="latitude",
    lng_col="longitude",
)
train_data_2_24_new_subset_h3.reset_index(inplace=True)
train_data_2_24_new_subset_h3.drop_duplicates(subset=["h3_04"], inplace=True)

train_data_2_24_old_subset_h3 = train_data_2_24_old_subset.h3.geo_to_h3(
    resolution=4,
    lat_col="latitude",
    lng_col="longitude",
)
train_data_2_24_old_subset_h3.reset_index(inplace=True)
train_data_2_24_old_subset_h3.drop_duplicates(subset=["h3_04"], inplace=True)

In [53]:
new_occurred = list(train_data_2_24_new_subset_h3.h3_04.values)
old_occurred = list(train_data_2_24_old_subset_h3.h3_04.values)
true_new = list(set(new_occurred) - set(old_occurred))
true_new

[]

In [54]:
stable_negatives = 0
stable_positives = 0
discovery_misses = 0
discovery_hits = 0
pred_not_found = 0
found_not_pred = 0

for cell in tqdm(dfh3.index.values):
    in_pred = cell in predicted_cells
    in_new = cell in true_new
    in_old = cell in old_occurred

    if in_new:
        if in_pred:
            discovery_hits += 1
        else:
            discovery_misses += 1
    elif in_old:
        if in_pred:
            stable_positives += 1
        else:
            found_not_pred += 1
    else:
        if in_pred:
            pred_not_found += 1
        else:
            stable_negatives += 1
    if cell not in predicted_cells and cell not in new_occurred:
        stable_negatives += 1

print(f"{stable_negatives} stable negatives")
print(f"{stable_positives} stable positives")
print()
print(f"{pred_not_found} expected but no occurrence")
print(f"{found_not_pred} occurence but not expected")
print()
print(f"{discovery_hits} discovery hits")
print(f"{discovery_misses} discovery misses")

  0%|          | 0/288122 [00:00<?, ?it/s]

575948 stable negatives
71 stable positives

66 expected but no occurrence
17 occurence but not expected

0 discovery hits
0 discovery misses
